# Correct Disentanglement Probes for Torus Longitudinal Velocity Models

This notebook replaces the earlier norm-only velocity AUC diagnostic with a proper held-out external probe evaluation.

The core question is: after the INR and flow are trained, does each velocity component contain only the information it is supposed to contain?

The protocol is:

1. Load the same three selected models used in the latest best-3 report.
2. Use saved train subject anchors for train subjects.
3. Optimize one baseline-only latent for every test subject and every model, then cache those latents.
4. Compute instantaneous component velocities at each subject's baseline age: `s=t=baseline_age_norm`.
5. Train external linear probes on train subjects and evaluate them on held-out test subjects.
6. Save all plots and explanation pages into an indexed HTML report.

Important: norm-AUC is kept only as a secondary diagnostic. The main disentanglement result is the held-out vector-probe performance.

In [1]:
# Cell 1: configuration, imports, and HTML report helpers.
# This cell intentionally mirrors the latest HTML-report style: every important output is saved as an HTML page,
# and the notebook itself does not need to display large inline figures.
import html as html_lib
import json
import math
import random
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegressionCV, RidgeCV
from sklearn.metrics import balanced_accuracy_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path('/home/jakaria/INR/Deep3DComp')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import deep_sdf.data
import reconstruct
from networks.deep_sdf_decoder import Decoder
from networks.longitudinal_disentangled_flow_64_128_64_adv import build_temporal_flow as build_subset128_flow
from networks.longitudinal_flow_64_160_32_pred_dx import build_temporal_flow as build_subset160_flow
from networks.longitudinal_additive_flow import build_temporal_flow as build_additive_flow

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

GPU_ID = 0
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
DEVICE = torch.device(f'cuda:{GPU_ID}' if torch.cuda.is_available() else 'cpu')

# Test latent optimization is the only potentially slow part.
# The optimized test latents are cached under OUTPUT_DIR/cache, so rerunning the notebook should be much faster.
TEST_LATENT_ITERS = 500
TEST_LATENT_SAMPLES = 8192
TEST_LATENT_LR = 5e-4
TEST_LATENT_INIT = 'train_mean_std'
MAX_TEST_SUBJECTS = None  # set to a small integer for quick debugging; keep None for the real report.

OUTPUT_DIR = ROOT / 'analysis_torus_correct_disentanglement_probes'
FIGURE_DIR = OUTPUT_DIR / 'figures'
CACHE_DIR = OUTPUT_DIR / 'cache'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SAVE_INTERACTIVE_HTML = True
SHOW_INLINE_PLOTLY = False
FIGURE_MANIFEST = {}

MODEL_COLORS = {
    'Subset 64/128/64 best': '#C44536',
    'Subset 64/160/32 E2 best saved': '#147D92',
    'Additive strong best': '#D97706',
}
SHORT_NAMES = {
    'Subset 64/128/64 best': '64/128/64',
    'Subset 64/160/32 E2 best saved': '64/160/32',
    'Additive strong best': 'additive',
}
COMPONENT_ORDER = ['age', 'disease_raw', 'disease_predicted_soft', 'disease_oracle_gated', 'residual', 'total_raw']
COMPONENT_LABELS = {
    'age': 'age',
    'disease_raw': 'disease raw',
    'disease_predicted_soft': 'disease predicted',
    'disease_oracle_gated': 'disease oracle',
    'residual': 'residual',
    'total_raw': 'total raw',
}


def figure_slug(value):
    text = re.sub(r'[^A-Za-z0-9._-]+', '_', str(value)).strip('_')
    return text or 'figure'


def register_output(path, title, category, description=''):
    path = Path(path)
    FIGURE_MANIFEST[str(path)] = {
        'path': path,
        'title': str(title),
        'category': str(category),
        'description': str(description),
    }
    print('Saved output:', path)


def html_page(title, body, description=''):
    desc = f"<p class='note'>{html_lib.escape(str(description))}</p>" if description else ''
    return f"""<!doctype html>
<html><head><meta charset='utf-8'><title>{html_lib.escape(str(title))}</title>
<style>
body{{font-family:Arial,sans-serif;max-width:1180px;margin:32px auto;padding:0 22px;color:#202020}}
h1{{margin-bottom:8px}} h2{{margin-top:28px}} p{{line-height:1.55}} li{{margin:8px 0}}
.note{{background:#edf6f5;border-left:4px solid #147d92;padding:12px 14px;border-radius:6px}}
.warn{{background:#fff7ed;border-left:4px solid #d97706;padding:12px 14px;border-radius:6px}}
.bad{{background:#fef2f2;border-left:4px solid #c44536;padding:12px 14px;border-radius:6px}}
.good{{background:#f0fdf4;border-left:4px solid #2f855a;padding:12px 14px;border-radius:6px}}
code{{background:#f3f3f3;padding:2px 5px;border-radius:3px}}
table{{border-collapse:collapse;margin:16px 0;width:100%}} th,td{{border:1px solid #ddd;padding:8px;text-align:left}} th{{background:#f5f5f5}}
a{{color:#0b5cad;text-decoration:none}} a:hover{{text-decoration:underline}}
</style></head><body><h1>{html_lib.escape(str(title))}</h1>{desc}{body}</body></html>"""


def save_text_page(filename, title, category, body_html, description=''):
    html_path = FIGURE_DIR / f'{figure_slug(filename)}.html'
    html_path.write_text(html_page(title, body_html, description), encoding='utf-8')
    register_output(html_path, title, category, description)
    return html_path


def save_plotly_figure(fig, filename, title, category, description='', show_inline=False):
    html_path = FIGURE_DIR / f'{figure_slug(filename)}.html'
    fig.write_html(html_path, include_plotlyjs='directory', full_html=True, auto_open=False)
    register_output(html_path, title, category, description)
    if SHOW_INLINE_PLOTLY and show_inline:
        fig.show()
    return html_path


def write_figure_index(report_title='Correct Disentanglement Probe Report'):
    entries = sorted(FIGURE_MANIFEST.values(), key=lambda item: (item['category'], item['title'], str(item['path'])))
    groups = {}
    for item in entries:
        groups.setdefault(item['category'], []).append(item)
    chunks = [
        "<!doctype html><html><head><meta charset='utf-8'>",
        f"<title>{html_lib.escape(report_title)}</title>",
        "<style>body{font-family:Arial,sans-serif;max-width:1150px;margin:32px auto;padding:0 20px;color:#202020}",
        "h1{margin-bottom:8px}h2{margin-top:30px}li{margin:10px 0}a{color:#0b5cad;text-decoration:none}",
        "a:hover{text-decoration:underline}.desc{color:#555;font-size:14px}</style></head><body>",
        f"<h1>{html_lib.escape(report_title)}</h1>",
        "<p>This report uses held-out external probes on velocity component vectors. Norm-only AUC is shown only as a diagnostic.</p>",
        f"<p><strong>Output directory:</strong> <code>{html_lib.escape(str(FIGURE_DIR))}</code></p>",
    ]
    for category, items in groups.items():
        chunks.append(f"<h2>{html_lib.escape(category)}</h2><ul>")
        for item in items:
            relative = item['path'].relative_to(FIGURE_DIR)
            desc = html_lib.escape(item.get('description', ''))
            chunks.append(
                f"<li><a href='{relative.as_posix()}'>{html_lib.escape(item['title'])}</a>"
                f"<br><span class='desc'>{desc}</span></li>"
            )
        chunks.append('</ul>')
    chunks.append('</body></html>')
    index_path = FIGURE_DIR / 'index.html'
    index_path.write_text(''.join(chunks), encoding='utf-8')
    print('Figure index:', index_path)
    return index_path

print('Device:', DEVICE)
print('Output folder:', FIGURE_DIR)

Device: cuda:0
Output folder: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures


In [2]:
# Cell 2: explicit reading guide for the corrected metric.
# This page is saved before any numeric result, so the report is clear about what is and is not being measured.
save_text_page(
    '00_correct_disentanglement_protocol',
    'Correct disentanglement protocol',
    'Explanation',
    """
    <p>The previous norm-AUC analysis was only a speed diagnostic. This report uses external probes on the full velocity component vectors.</p>
    <h2>Main test</h2>
    <p>For each subject, compute instantaneous component velocities at baseline age:</p>
    <p><code>v_age, v_disease, v_residual = flow_components(z0, s=t=baseline_age_norm)</code></p>
    <p>Then train probes on train subjects and evaluate on held-out test subjects.</p>
    <h2>Desired behavior</h2>
    <ul>
      <li><code>diagnosis from v_age</code> should fail; AUC near 0.5.</li>
      <li><code>diagnosis from v_residual</code> should fail; AUC near 0.5.</li>
      <li><code>diagnosis from v_disease_raw</code> should succeed only if the disease branch contains label-free disease information.</li>
      <li><code>age from v_age</code> should succeed.</li>
      <li><code>age from v_disease_raw</code> and <code>age from v_residual</code> should be low if those branches are age-clean.</li>
    </ul>
    <p class='warn'><strong>Oracle warning:</strong> if the true disease label is fed to the model, disease velocity can become zero for healthy subjects by construction. That is not label-free disentanglement.</p>
    """,
    'Defines the corrected held-out external-probe protocol and flags oracle-gated disease as non-label-free.',
)

Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/00_correct_disentanglement_protocol.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/00_correct_disentanglement_protocol.html')

In [3]:
# Cell 3: load labels, train/test splits, models, and saved train subject anchors.
# Train anchors are the learned subject anchors from each experiment. Test anchors will be optimized in the next cell.
BASE = ROOT / 'examples' / 'Torus_subset_100_id_age_progression'
SUBSET128_DIR = BASE / 'longitudinal_age_disease_conditioned_cocycle_shape_pair_loss_multiple_pairs_disentangled_velocity_64_128_64_binary_margin_adv'
SUBSET160_DIR = BASE / 'velocity_64_160_32_baseline_dx'
ADDITIVE_DIR = BASE / 'longitudinal_age_disease_additive_velocity_disentanglement_strong_residual_invariance_adv'
LABELS_PATH = Path('/home/jakaria/torus_creation/torus_age_disease_longitudinal_100ids_5tp_flat/labels.pt')

MODEL_CONFIGS = {
    'subset64_128_best': {
        'label': 'Subset 64/128/64 best',
        'short_label': '64/128/64',
        'kind': 'subset128',
        'exp_dir': SUBSET128_DIR,
        'checkpoint': '500',
    },
    'subset64_160_best_saved': {
        'label': 'Subset 64/160/32 E2 best saved',
        'short_label': '64/160/32',
        'kind': 'subset160',
        'exp_dir': SUBSET160_DIR,
        'checkpoint': '1000',
    },
    'additive_best': {
        'label': 'Additive strong best',
        'short_label': 'additive',
        'kind': 'additive',
        'exp_dir': ADDITIVE_DIR,
        'checkpoint': '1000',
    },
}


def load_label_dataframe(labels_path):
    obj = torch.load(labels_path, map_location='cpu')
    rows = []
    for scan_id, payload in obj.items():
        row = {'scan_id': str(scan_id)}
        row.update(payload)
        rows.append(row)
    df = pd.DataFrame(rows)
    df['sid'] = df['scan_id'].str.extract(r'ID_(\d+)_t').astype(int)
    df['tp'] = df['scan_id'].str.extract(r'_t(\d+)').astype(int)
    return df.sort_values(['sid', 'tp']).reset_index(drop=True)


def ordered_subjects_from_split(split_path):
    split = json.loads(Path(split_path).read_text())
    seen = []
    for item in split:
        sid = int(re.search(r'ID_(\d+)_t', Path(str(item)).stem).group(1))
        if sid not in seen:
            seen.append(sid)
    return seen


def baseline_table_for_subjects(label_df, subject_ids, split_name):
    base = label_df[label_df['tp'] == 0].set_index('sid')
    rows = []
    for sid in subject_ids:
        row = base.loc[int(sid)].to_dict()
        row.update({'sid': int(sid), 'split': split_name})
        rows.append(row)
    out = pd.DataFrame(rows)
    out['age_norm'] = out['age_norm'].astype(float)
    out['age'] = out['age'].astype(float)
    out['diagnosis'] = out['diagnosis'].astype(int)
    return out


def checkpoint_path(exp_dir, subdir, checkpoint):
    name = str(checkpoint)
    if not name.endswith('.pth'):
        name += '.pth'
    path = Path(exp_dir) / subdir / name
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def strip_module_prefix(state):
    return {k.removeprefix('module.'): v for k, v in state.items()}


def latent_weights(payload):
    lat = payload.get('latent_codes_state_dict', payload.get('latent_codes'))
    if isinstance(lat, dict):
        lat = lat['weight']
    if lat.ndim == 3 and lat.shape[1] == 1:
        lat = lat[:, 0]
    return lat.detach().float()


def flow_builder(kind):
    if kind == 'subset128':
        return build_subset128_flow
    if kind == 'subset160':
        return build_subset160_flow
    if kind == 'additive':
        return build_additive_flow
    raise ValueError(kind)


def load_model(config):
    exp_dir = Path(config['exp_dir'])
    specs = json.loads((exp_dir / 'specs.json').read_text())
    decoder = Decoder(int(specs['CodeLength']), **specs['NetworkSpecs']).to(DEVICE)
    flow = flow_builder(config['kind'])(
        specs,
        int(specs['CodeLength']),
        list(specs.get('FlowHiddenDims', [256, 256])),
        age_condition_dim=int(specs.get('AgeConditionDim', 0)),
    ).to(DEVICE)
    model_payload = torch.load(checkpoint_path(exp_dir, 'ModelParameters', config['checkpoint']), map_location=DEVICE)
    decoder.load_state_dict(strip_module_prefix(model_payload['model_state_dict']))
    flow.load_state_dict(strip_module_prefix(model_payload['flow_state_dict']))
    decoder.eval(); flow.eval()
    latent_payload = torch.load(checkpoint_path(exp_dir, 'LatentCodes', config['checkpoint']), map_location='cpu')
    weights = latent_weights(latent_payload)
    train_subjects = ordered_subjects_from_split(specs['TrainSplit'])
    test_subjects = ordered_subjects_from_split(specs['TestSplit'])
    if len(weights) != len(train_subjects):
        raise RuntimeError(f"Latent count mismatch for {config['label']}: {len(weights)} latents vs {len(train_subjects)} train subjects")
    return {
        **config,
        'specs': specs,
        'decoder': decoder,
        'flow': flow,
        'latent_weights': weights,
        'latent_mean': weights.mean(dim=0, keepdim=True),
        'latent_std': weights.std(dim=0, keepdim=True).clamp_min(1e-4),
        'train_subjects': train_subjects,
        'test_subjects': test_subjects,
        'epoch': int(model_payload.get('epoch', -1)),
    }

LABEL_DF = load_label_dataframe(LABELS_PATH)
MODELS = {name: load_model(cfg) for name, cfg in MODEL_CONFIGS.items()}

TRAIN_TABLE = baseline_table_for_subjects(LABEL_DF, next(iter(MODELS.values()))['train_subjects'], 'train')
TEST_TABLE = baseline_table_for_subjects(LABEL_DF, next(iter(MODELS.values()))['test_subjects'], 'test')
if MAX_TEST_SUBJECTS is not None:
    TEST_TABLE = TEST_TABLE.head(int(MAX_TEST_SUBJECTS)).copy()

summary_rows = []
for model in MODELS.values():
    summary_rows.append({
        'model': model['label'],
        'checkpoint': model['checkpoint'],
        'train_subjects': len(model['train_subjects']),
        'test_subjects': len(TEST_TABLE),
        'train_latents': tuple(model['latent_weights'].shape),
        'has_anchor_disease_head': callable(getattr(model['flow'], 'anchor_disease_probability', None)),
    })
MODEL_SUMMARY = pd.DataFrame(summary_rows)
MODEL_SUMMARY.to_csv(OUTPUT_DIR / 'model_summary.csv', index=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Subject counts', 'Baseline diagnosis balance'))
for model in MODEL_SUMMARY['model']:
    fig.add_trace(go.Bar(x=['train', 'test'], y=[len(TRAIN_TABLE), len(TEST_TABLE)], name=model, marker_color=MODEL_COLORS.get(model)), row=1, col=1)
for split_name, table, color in [('train', TRAIN_TABLE, '#147D92'), ('test', TEST_TABLE, '#D97706')]:
    counts = table['diagnosis'].value_counts().reindex([0, 1], fill_value=0)
    fig.add_trace(go.Bar(x=['healthy', 'diseased'], y=counts.values, name=split_name, marker_color=color), row=1, col=2)
fig.update_layout(title='Loaded models and baseline subject splits', width=1150, height=480, barmode='group')
fig.update_yaxes(title='count')
save_plotly_figure(
    fig,
    '01_model_and_split_summary',
    'Loaded models and split summary',
    'Setup',
    'Train anchors are saved subject latents. Test anchors are optimized from baseline scan only and cached by this notebook.',
)
print(MODEL_SUMMARY)

Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/01_model_and_split_summary.html
                            model checkpoint  train_subjects  test_subjects  \
0           Subset 64/128/64 best        500              80             15   
1  Subset 64/160/32 E2 best saved       1000              80             15   
2            Additive strong best       1000              80             15   

  train_latents  has_anchor_disease_head  
0     (80, 256)                    False  
1     (80, 256)                     True  
2     (80, 256)                    False  


In [4]:
# Cell 4: optimize or load cached baseline-only test latents.
# This is the only part that can take several minutes on first run. The cache key includes model name, checkpoint, and optimization settings.
def sdf_npz_path(specs, scan_id):
    return Path(specs['DataSource']) / f'{scan_id}.npz'


def load_sdf_samples_for_scan(specs, scan_id):
    path = sdf_npz_path(specs, scan_id)
    if not path.exists():
        raise FileNotFoundError(path)
    samples = deep_sdf.data.read_sdf_samples_into_ram(str(path))
    # Shuffle once before reconstruction to avoid always seeing the same contiguous chunks.
    samples[0] = samples[0][torch.randperm(samples[0].shape[0])]
    samples[1] = samples[1][torch.randperm(samples[1].shape[0])]
    return samples


def optimize_single_baseline_latent(model, scan_id):
    specs = model['specs']
    samples = load_sdf_samples_for_scan(specs, scan_id)
    latent_size = int(specs['CodeLength'])
    clamp_dist = float(specs.get('ClampingDistance', 0.1))
    code_reg_lambda = float(specs.get('CodeRegularizationLambda', 0.0)) if bool(specs.get('CodeRegularization', False)) else 0.0
    code_bound = specs.get('CodeBound', None)
    if TEST_LATENT_INIT == 'train_mean_std':
        stat = (model['latent_mean'].to(DEVICE), model['latent_std'].to(DEVICE))
    else:
        stat = 0.01
    loss_hist, latent = reconstruct.reconstruct(
        model['decoder'],
        int(TEST_LATENT_ITERS),
        latent_size,
        samples,
        stat,
        clamp_dist,
        num_samples=int(TEST_LATENT_SAMPLES),
        lr=float(TEST_LATENT_LR),
        l2reg=False,
        code_reg_lambda=code_reg_lambda,
        code_reg_type='l2_sq',
        code_bound=code_bound,
        return_loss_hist=True,
    )
    return latent.detach().cpu().reshape(1, -1), float(loss_hist[-1]) if loss_hist else float('nan')


def test_latent_cache_path(model):
    stem = f"{model['short_label']}_ckpt_{model['checkpoint']}_iters_{TEST_LATENT_ITERS}_samples_{TEST_LATENT_SAMPLES}"
    return CACHE_DIR / f'{figure_slug(stem)}_test_baseline_latents.pt'


def load_or_optimize_test_latents(model, test_table):
    cache_path = test_latent_cache_path(model)
    expected_sids = [int(x) for x in test_table['sid'].tolist()]
    if cache_path.exists():
        payload = torch.load(cache_path, map_location='cpu')
        if payload.get('subject_ids') == expected_sids:
            print(f"Loaded cached test latents for {model['label']}: {cache_path}")
            return payload['latents'].float(), pd.DataFrame(payload['rows'])
        print(f"Cache subject mismatch, recomputing: {cache_path}")

    latents = []
    rows = []
    for _, row in test_table.iterrows():
        scan_id = str(row['scan_id'])
        sid = int(row['sid'])
        latent, loss = optimize_single_baseline_latent(model, scan_id)
        latents.append(latent)
        rows.append({
            'model': model['label'],
            'sid': sid,
            'scan_id': scan_id,
            'diagnosis': int(row['diagnosis']),
            'age': float(row['age']),
            'age_norm': float(row['age_norm']),
            'baseline_reconstruction_loss': loss,
        })
        print(f"{model['short_label']} test SID {sid:03d}: baseline latent loss={loss:.6g}")

    latents = torch.cat(latents, dim=0).float()
    payload = {
        'subject_ids': expected_sids,
        'latents': latents,
        'rows': rows,
        'settings': {
            'iters': TEST_LATENT_ITERS,
            'samples': TEST_LATENT_SAMPLES,
            'lr': TEST_LATENT_LR,
            'init': TEST_LATENT_INIT,
        },
    }
    torch.save(payload, cache_path)
    print(f"Saved cached test latents for {model['label']}: {cache_path}")
    return latents, pd.DataFrame(rows)

TEST_LATENTS = {}
TEST_LATENT_ROWS = []
for key, model in MODELS.items():
    latents, rows = load_or_optimize_test_latents(model, TEST_TABLE)
    TEST_LATENTS[key] = latents
    TEST_LATENT_ROWS.append(rows)
TEST_LATENT_DF = pd.concat(TEST_LATENT_ROWS, ignore_index=True)
TEST_LATENT_DF.to_csv(OUTPUT_DIR / 'test_baseline_latent_reconstruction_losses.csv', index=False)

fig = go.Figure()
for model_label, sub in TEST_LATENT_DF.groupby('model'):
    fig.add_trace(go.Box(y=sub['baseline_reconstruction_loss'], name=SHORT_NAMES.get(model_label, model_label), marker_color=MODEL_COLORS.get(model_label)))
fig.update_layout(
    title='Baseline-only test latent optimization losses',
    yaxis_title='final baseline SDF L1 + latent regularization loss',
    width=950,
    height=500,
)
save_plotly_figure(
    fig,
    '02_test_baseline_latent_optimization_losses',
    'Test baseline latent optimization losses',
    'Setup',
    'Each test latent is optimized from only the baseline scan. These latents are then used for held-out velocity probes.',
)

64/128/64 test SID 004: baseline latent loss=0.00243294
64/128/64 test SID 011: baseline latent loss=0.00128511
64/128/64 test SID 013: baseline latent loss=0.00436066
64/128/64 test SID 017: baseline latent loss=0.0017404
64/128/64 test SID 027: baseline latent loss=0.00733125
64/128/64 test SID 028: baseline latent loss=0.00189711
64/128/64 test SID 029: baseline latent loss=0.00479743
64/128/64 test SID 031: baseline latent loss=0.00144842
64/128/64 test SID 054: baseline latent loss=0.0041876
64/128/64 test SID 064: baseline latent loss=0.00082334
64/128/64 test SID 069: baseline latent loss=0.00231587
64/128/64 test SID 075: baseline latent loss=0.0012018
64/128/64 test SID 086: baseline latent loss=0.00201388
64/128/64 test SID 088: baseline latent loss=0.000882334
64/128/64 test SID 097: baseline latent loss=0.00309336
Saved cached test latents for Subset 64/128/64 best: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/cache/64_128_64_ckpt_500_iters_500

PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/02_test_baseline_latent_optimization_losses.html')

In [5]:
# Cell 5: compute instantaneous velocity component vectors for train and test subjects.
# Main components are computed with a raw/no-label disease gate, so the true disease label is not injected into the feature.
# Oracle-gated disease is computed separately only to show how much label injection can inflate apparent disentanglement.
def zero_pad_subset(model, comp, block):
    flow = model['flow']
    z = comp.new_zeros(comp.shape[0], flow.age_dim + flow.disease_dim + flow.residual_dim)
    if block == 'age':
        z[:, :flow.age_dim] = comp
    elif block == 'disease':
        z[:, flow.age_dim:flow.age_dim + flow.disease_dim] = comp
    elif block == 'residual':
        z[:, flow.age_dim + flow.disease_dim:] = comp
    else:
        raise ValueError(block)
    return z


def component_vectors_for_conditions(model, z, age_norm, diagnosis_true=None):
    z = z.to(DEVICE).float()
    age_norm = torch.as_tensor(age_norm, dtype=z.dtype, device=DEVICE).view(-1, 1)
    ones = torch.ones((z.shape[0], 1), dtype=z.dtype, device=DEVICE)
    zeros = torch.zeros((z.shape[0], 1), dtype=z.dtype, device=DEVICE)
    true_q = None
    if diagnosis_true is not None:
        true_q = torch.as_tensor(diagnosis_true, dtype=z.dtype, device=DEVICE).view(-1, 1)

    with torch.no_grad():
        raw = model['flow'].velocity_components(z, age_norm, age_norm, age_cond=ones)
        zero = model['flow'].velocity_components(z, age_norm, age_norm, age_cond=zeros)
        oracle = model['flow'].velocity_components(z, age_norm, age_norm, age_cond=true_q) if true_q is not None else None
        pred = None
        if callable(getattr(model['flow'], 'anchor_disease_probability', None)):
            q_pred = model['flow'].anchor_disease_probability(z).view(-1, 1)
            pred = model['flow'].velocity_components(z, age_norm, age_norm, age_cond=q_pred)
        else:
            q_pred = None

    if model['kind'] in ('subset128', 'subset160'):
        age = zero_pad_subset(model, raw['age'], 'age')
        disease_raw = zero_pad_subset(model, raw.get('disease_raw', raw['disease']), 'disease')
        residual = zero_pad_subset(model, raw['residual'], 'residual')
        disease_oracle = zero_pad_subset(model, oracle['disease'], 'disease') if oracle is not None else None
        disease_pred = zero_pad_subset(model, pred['disease'], 'disease') if pred is not None else None
    else:
        age = raw['age']
        disease_raw = raw['disease']
        residual = raw['residual']
        disease_oracle = oracle['disease'] if oracle is not None else None
        disease_pred = pred['disease'] if pred is not None else None

    out = {
        'age': age.detach().cpu(),
        'disease_raw': disease_raw.detach().cpu(),
        'residual': residual.detach().cpu(),
        'total_raw': (age + disease_raw + residual).detach().cpu(),
    }
    if disease_oracle is not None:
        out['disease_oracle_gated'] = disease_oracle.detach().cpu()
    if disease_pred is not None:
        out['disease_predicted_soft'] = disease_pred.detach().cpu()
        out['predicted_disease_probability'] = q_pred.detach().cpu()
    return out


def build_train_latent_table(model):
    table = baseline_table_for_subjects(LABEL_DF, model['train_subjects'], 'train')
    table = table.reset_index(drop=True)
    z = model['latent_weights'].float()
    return table, z


def build_feature_store():
    store = {}
    meta_rows = []
    for key, model in MODELS.items():
        train_table, z_train = build_train_latent_table(model)
        test_table = TEST_TABLE.reset_index(drop=True).copy()
        z_test = TEST_LATENTS[key].float()
        for split_name, table, z in [('train', train_table, z_train), ('test', test_table, z_test)]:
            comps = component_vectors_for_conditions(
                model,
                z,
                table['age_norm'].to_numpy(dtype=float),
                diagnosis_true=table['diagnosis'].to_numpy(dtype=float),
            )
            store[(key, split_name)] = {'table': table, 'components': comps}
            for _, row in table.iterrows():
                meta_rows.append({
                    'model_key': key,
                    'model': model['label'],
                    'split': split_name,
                    'sid': int(row['sid']),
                    'scan_id': str(row['scan_id']),
                    'age': float(row['age']),
                    'age_norm': float(row['age_norm']),
                    'diagnosis': int(row['diagnosis']),
                    'has_predicted_disease_component': 'disease_predicted_soft' in comps,
                })
    return store, pd.DataFrame(meta_rows)

FEATURE_STORE, FEATURE_META = build_feature_store()
FEATURE_META.to_csv(OUTPUT_DIR / 'velocity_feature_subject_metadata.csv', index=False)
print('Velocity feature metadata rows:', len(FEATURE_META))

Velocity feature metadata rows: 285


In [6]:
# Cell 6: external probe definitions.
# Probes are trained on train subjects and evaluated on held-out test subjects.
# The vector input is the component velocity itself, not just its norm.
def safe_auc(y_true, score):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, score))


def norm_auc(y_true, x):
    score = np.linalg.norm(np.asarray(x, dtype=float), axis=1)
    return safe_auc(y_true, score)


def pca_components_for(x_train):
    n, d = x_train.shape
    return int(max(1, min(16, d, n - 2)))


def disease_probe(x_train, y_train, x_test, y_test):
    x_train = np.asarray(x_train, dtype=np.float32)
    x_test = np.asarray(x_test, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=int)
    y_test = np.asarray(y_test, dtype=int)
    n_comp = pca_components_for(x_train)
    cv_splits = min(5, int(np.bincount(y_train).min()))
    if cv_splits < 2:
        return {'train_auc': np.nan, 'test_auc': np.nan, 'test_bal_acc': np.nan, 'note': 'not enough classes'}
    model = Pipeline([
        ('scale', StandardScaler()),
        ('pca', PCA(n_components=n_comp, random_state=SEED)),
        ('clf', LogisticRegressionCV(
            Cs=np.logspace(-3, 1, 8),
            cv=StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=SEED),
            scoring='roc_auc',
            class_weight='balanced',
            max_iter=5000,
            solver='liblinear',
            random_state=SEED,
        )),
    ])
    model.fit(x_train, y_train)
    train_prob = model.predict_proba(x_train)[:, 1]
    test_prob = model.predict_proba(x_test)[:, 1]
    test_pred = (test_prob >= 0.5).astype(int)
    return {
        'train_auc': safe_auc(y_train, train_prob),
        'test_auc': safe_auc(y_test, test_prob),
        'test_bal_acc': float(balanced_accuracy_score(y_test, test_pred)),
        'pca_dim': n_comp,
        'note': '',
    }


def age_probe(x_train, y_train, x_test, y_test):
    x_train = np.asarray(x_train, dtype=np.float32)
    x_test = np.asarray(x_test, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=float)
    y_test = np.asarray(y_test, dtype=float)
    n_comp = pca_components_for(x_train)
    model = Pipeline([
        ('scale', StandardScaler()),
        ('pca', PCA(n_components=n_comp, random_state=SEED)),
        ('reg', RidgeCV(alphas=np.logspace(-4, 4, 25), cv=KFold(n_splits=5, shuffle=True, random_state=SEED))),
    ])
    model.fit(x_train, y_train)
    pred_train = model.predict(x_train)
    pred_test = model.predict(x_test)
    if np.std(pred_test) < 1e-12 or np.std(y_test) < 1e-12:
        corr = np.nan
    else:
        corr = float(np.corrcoef(y_test, pred_test)[0, 1])
    return {
        'train_r2': float(r2_score(y_train, pred_train)),
        'test_r2': float(r2_score(y_test, pred_test)),
        'test_mae_years': float(mean_absolute_error(y_test, pred_test)),
        'test_corr': corr,
        'pca_dim': n_comp,
        'note': '',
    }


def available_components(train_components, test_components):
    comps = []
    for comp in COMPONENT_ORDER:
        if comp in train_components and comp in test_components:
            comps.append(comp)
    return comps

probe_rows = []
norm_rows = []
for key, model in MODELS.items():
    train = FEATURE_STORE[(key, 'train')]
    test = FEATURE_STORE[(key, 'test')]
    train_table = train['table']
    test_table = test['table']
    y_train_dx = train_table['diagnosis'].to_numpy(dtype=int)
    y_test_dx = test_table['diagnosis'].to_numpy(dtype=int)
    y_train_age = train_table['age'].to_numpy(dtype=float)
    y_test_age = test_table['age'].to_numpy(dtype=float)
    for comp in available_components(train['components'], test['components']):
        x_train = train['components'][comp].numpy()
        x_test = test['components'][comp].numpy()
        dx = disease_probe(x_train, y_train_dx, x_test, y_test_dx)
        probe_rows.append({
            'model_key': key,
            'model': model['label'],
            'component': comp,
            'target': 'diagnosis',
            **dx,
        })
        ag = age_probe(x_train, y_train_age, x_test, y_test_age)
        probe_rows.append({
            'model_key': key,
            'model': model['label'],
            'component': comp,
            'target': 'baseline_age_years',
            **ag,
        })
        norm_rows.append({
            'model_key': key,
            'model': model['label'],
            'component': comp,
            'diagnosis_norm_auc_train': norm_auc(y_train_dx, x_train),
            'diagnosis_norm_auc_test': norm_auc(y_test_dx, x_test),
            'mean_norm_train': float(np.linalg.norm(x_train, axis=1).mean()),
            'mean_norm_test': float(np.linalg.norm(x_test, axis=1).mean()),
        })

PROBE_DF = pd.DataFrame(probe_rows)
NORM_DIAG_DF = pd.DataFrame(norm_rows)
PROBE_DF.to_csv(OUTPUT_DIR / 'correct_external_probe_metrics.csv', index=False)
NORM_DIAG_DF.to_csv(OUTPUT_DIR / 'norm_only_diagnostic_metrics.csv', index=False)
print('Probe rows:', len(PROBE_DF))

/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning:

Ill-conditioned matrix (rcond=5.85327e-08): result may not be accurate.

/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning:

Ill-conditioned matrix (rcond=5.56652e-08): result may not be accurate.

/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning:

Ill-conditioned matrix (rcond=4.9771e-08): result may not be accurate.

/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning:

Ill-conditioned matrix (rcond=4.57395e-08): result may not be accurate.

/home/jakaria/anaconda3/envs/inr_sdf/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning:

Ill-conditioned matrix (rcond=4.71555e-08): result may not be accurate.

/home/jakaria/anaconda3/envs/inr_sdf/lib/pytho

Probe rows: 32


In [7]:
# Cell 7: plot the main corrected diagnosis disentanglement result.
# Interpretation: age/residual AUC near 0.5 is good; disease_raw high is useful only when no true label was injected.
dx = PROBE_DF[PROBE_DF['target'] == 'diagnosis'].copy()
dx['component_label'] = dx['component'].map(COMPONENT_LABELS).fillna(dx['component'])

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[MODELS[k]['short_label'] for k in MODELS.keys()],
    shared_yaxes=True,
)
for col, (key, model) in enumerate(MODELS.items(), start=1):
    sub = dx[dx['model_key'] == key].set_index('component').reindex(COMPONENT_ORDER).dropna(subset=['test_auc']).reset_index()
    colors = []
    for comp in sub['component']:
        if comp in ('age', 'residual'):
            colors.append('#2F855A')
        elif comp == 'disease_raw':
            colors.append('#147D92')
        elif comp == 'disease_predicted_soft':
            colors.append('#0B5CAD')
        elif comp == 'disease_oracle_gated':
            colors.append('#C44536')
        else:
            colors.append('#777777')
    fig.add_trace(go.Bar(
        x=sub['component'].map(COMPONENT_LABELS).fillna(sub['component']),
        y=sub['test_auc'],
        marker_color=colors,
        name=model['label'],
        showlegend=False,
        customdata=np.stack([sub['train_auc'], sub['test_bal_acc']], axis=1),
        hovertemplate='component=%{x}<br>test AUC=%{y:.3f}<br>train AUC=%{customdata[0]:.3f}<br>test bal acc=%{customdata[1]:.3f}<extra></extra>',
    ), row=1, col=col)
    fig.add_hline(y=0.5, line_dash='dash', line_color='black', row=1, col=col)
fig.update_layout(
    title='Correct diagnosis leakage probe: external classifier on full component vectors',
    width=1350,
    height=520,
    margin=dict(b=130),
)
fig.update_yaxes(title='held-out diagnosis AUC', range=[0, 1.05])
save_plotly_figure(
    fig,
    '03_correct_diagnosis_probe_auc',
    'Correct diagnosis probe AUC from component vectors',
    'Correct probes',
    'External classifiers are trained on train subject component vectors and evaluated on test subject component vectors. Age and residual should be near 0.5.',
)

Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/03_correct_diagnosis_probe_auc.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/03_correct_diagnosis_probe_auc.html')

In [8]:
# Cell 8: plot age leakage/predictability from each component.
# Interpretation: age component should predict baseline age. Disease/residual predicting age indicates time/age leakage into those branches.
age = PROBE_DF[PROBE_DF['target'] == 'baseline_age_years'].copy()
age['component_label'] = age['component'].map(COMPONENT_LABELS).fillna(age['component'])

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[f"{MODELS[k]['short_label']} R2" for k in MODELS.keys()] + [f"{MODELS[k]['short_label']} MAE" for k in MODELS.keys()],
    shared_yaxes=False,
)
for col, (key, model) in enumerate(MODELS.items(), start=1):
    sub = age[age['model_key'] == key].set_index('component').reindex(COMPONENT_ORDER).dropna(subset=['test_r2']).reset_index()
    x = sub['component'].map(COMPONENT_LABELS).fillna(sub['component'])
    fig.add_trace(go.Bar(x=x, y=sub['test_r2'], marker_color=MODEL_COLORS.get(model['label']), showlegend=False), row=1, col=col)
    fig.add_trace(go.Bar(x=x, y=sub['test_mae_years'], marker_color=MODEL_COLORS.get(model['label']), showlegend=False), row=2, col=col)
    fig.add_hline(y=0.0, line_dash='dash', line_color='black', row=1, col=col)
fig.update_layout(
    title='Age probe from component vectors: high age only desired for age component',
    width=1350,
    height=760,
    margin=dict(b=140),
)
fig.update_yaxes(title='held-out age R2', row=1)
fig.update_yaxes(title='held-out MAE years', row=2)
save_plotly_figure(
    fig,
    '04_correct_age_probe_from_components',
    'Correct age probe from component vectors',
    'Correct probes',
    'External regressors predict baseline age from each component vector. Disease and residual should not strongly predict age if they are clean.',
)

Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/04_correct_age_probe_from_components.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/04_correct_age_probe_from_components.html')

In [9]:
# Cell 9: compare the old norm-only diagnostic against the corrected vector-probe result.
# This directly shows why the previous metric was insufficient.
norm = NORM_DIAG_DF.copy()
dx_auc = dx[['model_key', 'model', 'component', 'test_auc']].rename(columns={'test_auc': 'vector_probe_test_auc'})
compare = norm.merge(dx_auc, on=['model_key', 'model', 'component'], how='left')
compare['component_label'] = compare['component'].map(COMPONENT_LABELS).fillna(compare['component'])
compare.to_csv(OUTPUT_DIR / 'norm_vs_vector_probe_comparison.csv', index=False)

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[MODELS[k]['short_label'] for k in MODELS.keys()],
    shared_yaxes=True,
)
for col, (key, model) in enumerate(MODELS.items(), start=1):
    sub = compare[compare['model_key'] == key].set_index('component').reindex(COMPONENT_ORDER).dropna(subset=['vector_probe_test_auc']).reset_index()
    x = sub['component'].map(COMPONENT_LABELS).fillna(sub['component'])
    fig.add_trace(go.Bar(x=x, y=sub['diagnosis_norm_auc_test'], name='norm-only AUC', marker_color='#BBBBBB', showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Bar(x=x, y=sub['vector_probe_test_auc'], name='vector-probe AUC', marker_color=MODEL_COLORS.get(model['label']), showlegend=(col == 1)), row=1, col=col)
    fig.add_hline(y=0.5, line_dash='dash', line_color='black', row=1, col=col)
fig.update_layout(
    title='Norm-only diagnostic versus corrected vector-probe diagnosis AUC',
    width=1350,
    height=540,
    barmode='group',
    margin=dict(b=130),
)
fig.update_yaxes(title='held-out diagnosis AUC', range=[0, 1.05])
save_plotly_figure(
    fig,
    '05_norm_only_vs_vector_probe_auc',
    'Norm-only diagnostic versus vector-probe AUC',
    'Metric comparison',
    'Norm-only AUC uses only speed magnitude. Vector-probe AUC uses the full component vector and is the corrected leakage test.',
)

Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/05_norm_only_vs_vector_probe_auc.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/05_norm_only_vs_vector_probe_auc.html')

In [10]:
# Cell 10: explicitly visualize oracle-gating inflation.
# If disease_oracle_gated is high while disease_raw is low, the disease separation is mostly caused by injecting the true label.
oracle_rows = []
for key, model in MODELS.items():
    sub = dx[(dx['model_key'] == key) & (dx['component'].isin(['disease_raw', 'disease_predicted_soft', 'disease_oracle_gated']))]
    for _, row in sub.iterrows():
        oracle_rows.append(row.to_dict())
ORACLE_DF = pd.DataFrame(oracle_rows)
ORACLE_DF.to_csv(OUTPUT_DIR / 'oracle_gating_warning_metrics.csv', index=False)

fig = go.Figure()
for comp, color in [('disease_raw', '#147D92'), ('disease_predicted_soft', '#0B5CAD'), ('disease_oracle_gated', '#C44536')]:
    sub = ORACLE_DF[ORACLE_DF['component'] == comp]
    if len(sub) == 0:
        continue
    fig.add_trace(go.Bar(
        x=sub['model'].map(SHORT_NAMES).fillna(sub['model']),
        y=sub['test_auc'],
        name=COMPONENT_LABELS.get(comp, comp),
        marker_color=color,
    ))
fig.add_hline(y=0.5, line_dash='dash', line_color='black')
fig.update_layout(
    title='Disease component AUC: raw versus predicted versus oracle-gated',
    xaxis_title='model',
    yaxis_title='held-out diagnosis AUC',
    width=1050,
    height=520,
    barmode='group',
)
fig.update_yaxes(range=[0, 1.05])
save_plotly_figure(
    fig,
    '06_oracle_gating_warning',
    'Oracle-gating warning for disease component AUC',
    'Metric comparison',
    'Oracle-gated disease uses the true label and can inflate apparent disentanglement. Raw and predicted-soft modes are more relevant for label-free claims.',
)

Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/06_oracle_gating_warning.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/06_oracle_gating_warning.html')

In [11]:
# Cell 11: compute a conservative model summary score from the corrected probes.
# This score is intentionally simple and transparent: reward disease information in disease_raw/predicted components and penalize diagnosis leakage in age/residual.
score_rows = []
for key, model in MODELS.items():
    sub = dx[dx['model_key'] == key].set_index('component')
    age_leak = float(sub.loc['age', 'test_auc']) if 'age' in sub.index else np.nan
    residual_leak = float(sub.loc['residual', 'test_auc']) if 'residual' in sub.index else np.nan
    disease_raw = float(sub.loc['disease_raw', 'test_auc']) if 'disease_raw' in sub.index else np.nan
    disease_pred = float(sub.loc['disease_predicted_soft', 'test_auc']) if 'disease_predicted_soft' in sub.index else np.nan
    # For leakage, distance from 0.5 is bad in either direction because AUC < 0.5 still means separability after sign flip.
    leak_penalty = np.nanmean([abs(age_leak - 0.5), abs(residual_leak - 0.5)])
    label_free_signal = np.nanmax([disease_raw, disease_pred])
    score = (label_free_signal - 0.5) - leak_penalty
    score_rows.append({
        'model_key': key,
        'model': model['label'],
        'age_diagnosis_auc': age_leak,
        'residual_diagnosis_auc': residual_leak,
        'disease_raw_auc': disease_raw,
        'disease_predicted_auc': disease_pred,
        'leak_penalty_abs_auc_from_0p5': leak_penalty,
        'corrected_disentanglement_score': score,
    })
SCORE_DF = pd.DataFrame(score_rows)
SCORE_DF.to_csv(OUTPUT_DIR / 'corrected_disentanglement_score.csv', index=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Corrected score', 'Leakage penalty'))
fig.add_trace(go.Bar(
    x=SCORE_DF['model'].map(SHORT_NAMES).fillna(SCORE_DF['model']),
    y=SCORE_DF['corrected_disentanglement_score'],
    marker_color=[MODEL_COLORS.get(m) for m in SCORE_DF['model']],
    showlegend=False,
), row=1, col=1)
fig.add_trace(go.Bar(
    x=SCORE_DF['model'].map(SHORT_NAMES).fillna(SCORE_DF['model']),
    y=SCORE_DF['leak_penalty_abs_auc_from_0p5'],
    marker_color=[MODEL_COLORS.get(m) for m in SCORE_DF['model']],
    showlegend=False,
), row=1, col=2)
fig.add_hline(y=0.0, line_dash='dash', line_color='black', row=1, col=1)
fig.update_layout(
    title='Conservative corrected disentanglement summary',
    width=1100,
    height=480,
)
fig.update_yaxes(title='higher is better', row=1, col=1)
fig.update_yaxes(title='lower is better', row=1, col=2)
save_plotly_figure(
    fig,
    '07_corrected_disentanglement_summary_score',
    'Corrected disentanglement summary score',
    'Summary',
    'This transparent score rewards label-free disease signal in disease components and penalizes diagnosis leakage from age/residual components.',
)

Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/07_corrected_disentanglement_summary_score.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/07_corrected_disentanglement_summary_score.html')

In [12]:
# Cell 12: final interpretation page generated from the actual computed metrics.
# The text is rule-based so it updates automatically when you rerun the notebook with new checkpoints.
def fmt(x):
    if pd.isna(x):
        return 'n/a'
    return f'{float(x):.3f}'

best_idx = SCORE_DF['corrected_disentanglement_score'].idxmax()
best_row = SCORE_DF.loc[best_idx]
rows_html = []
for _, row in SCORE_DF.sort_values('corrected_disentanglement_score', ascending=False).iterrows():
    rows_html.append(
        '<tr>'
        f"<td>{html_lib.escape(str(row['model']))}</td>"
        f"<td>{fmt(row['corrected_disentanglement_score'])}</td>"
        f"<td>{fmt(row['disease_raw_auc'])}</td>"
        f"<td>{fmt(row['disease_predicted_auc'])}</td>"
        f"<td>{fmt(row['age_diagnosis_auc'])}</td>"
        f"<td>{fmt(row['residual_diagnosis_auc'])}</td>"
        f"<td>{fmt(row['leak_penalty_abs_auc_from_0p5'])}</td>"
        '</tr>'
    )

body = f"""
<p>The best corrected score in this run is:</p>
<p class='good'><strong>{html_lib.escape(str(best_row['model']))}</strong>, score = <code>{fmt(best_row['corrected_disentanglement_score'])}</code>.</p>
<h2>How to read the score</h2>
<p>The score is not a publication metric by itself. It is a compact diagnostic:</p>
<ul>
  <li>Higher disease AUC from raw or predicted disease components is good.</li>
  <li>Diagnosis AUC far from 0.5 in age or residual components is leakage and is penalized.</li>
  <li>Oracle-gated disease is excluded from the score because it uses the true disease label.</li>
</ul>
<h2>Corrected metric table</h2>
<table>
<thead><tr><th>Model</th><th>score</th><th>disease raw AUC</th><th>disease predicted AUC</th><th>age diagnosis AUC</th><th>residual diagnosis AUC</th><th>leak penalty</th></tr></thead>
<tbody>{''.join(rows_html)}</tbody>
</table>
<h2>What this fixes</h2>
<p>The previous report used <code>||v_component||</code> AUC as if it were disentanglement. That was incomplete. This report trains held-out probes on full vectors, so it can detect information stored in vector direction, not only speed magnitude.</p>
<h2>Remaining caveat</h2>
<p>Disease and residual networks receive time inputs in the current architectures. Therefore age prediction from disease/residual components is a real leakage diagnostic, but it also reflects the architectural fact that those branches are allowed to be time-dependent. If the goal is strict age-free disease/residual, the architecture or losses must prevent time information from being encoded there.</p>
"""
save_text_page(
    '08_final_corrected_disentanglement_interpretation',
    'Final corrected disentanglement interpretation',
    'Summary',
    body,
    'Automatically generated interpretation from held-out external probe results.',
)

index_path = write_figure_index('Correct Torus Velocity Disentanglement Probe Report')
print('Open this report:', index_path)

Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/08_final_corrected_disentanglement_interpretation.html
Figure index: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/index.html
Open this report: /home/jakaria/INR/Deep3DComp/analysis_torus_correct_disentanglement_probes/figures/index.html
